# Токенізація

Мовна модель працює не з текстом, а з послідовністю чисел. Тому перед входом
у трансформер текст проходить важливе перетворення:

1. **токенізатор** ділить текст на токени та замінює їх ідентифікаторами;

У цій лабораторній ми пройдемо шлях від токенів-слів до BPE і WordPiece,
а потім побачимо, чому самих токенів недостатньо для трансформера. Усі алгоритми
реалізуємо з нуля, використовуючи лише стандартну бібліотеку Python.



**Запуск:** виконайте SETUP.cmd один раз, потім START_JUPYTER.cmd. Відкрийте цей файл і оберіть Restart Kernel → Run All. Окремий GPU та API-ключ не потрібні. Це структура студентського шаблону: реалізації заповнені, короткі пояснення додані нижче. Функції та тести перевірено; повний FLORES-прогін цієї компактної редакції ще не повторювали. Результати попереднього повного запуску збережені окремо в results/.

## 1. Токенізатор на рівні слів

Найпростіша ідея — вважати кожне слово окремим токеном. Спочатку приведемо
текст до нижнього регістру, відділимо розділові знаки й збережемо український
апостроф усередині слова.

```python
word_tokenize("Комп'ютер навчається!")
# ["комп'ютер", "навчається", "!"]
```

Така нормалізація зменшує словник: `Модель` і `модель` стають одним токеном.
Водночас ми втрачаємо частину інформації. У реальних токенізаторах правила
нормалізації є окремим важливим рішенням.



**На захисті:** токен - одиниця тексту; ID - її номер. lower() зменшує словник, але прибирає регістр. Деталі регулярного виразу по кожному символу можна опустити.

In [ ]:
import re
from collections import Counter


def word_tokenize(text):
    """Слова (з апострофами), числа та окремі знаки; нижній регістр."""
    return re.findall(r"[^\W\d_]+(?:['’ʼ][^\W\d_]+)*|\d+|[^\w\s]|_", text.lower())


def test_word_tokenize():
    assert word_tokenize("Модель читає текст.") == [
        "модель", "читає", "текст", "."
    ]
    assert word_tokenize("Комп'ютер і пам’ять") == [
        "комп'ютер", "і", "пам’ять"
    ]
    assert word_tokenize("GPT-2: 50 токенів!") == [
        "gpt", "-", "2", ":", "50", "токенів", "!"
    ]
    assert word_tokenize("") == []


if __name__ == "__main__":
    test_word_tokenize()
    print("✓ word_tokenize")

### Словник та проблема OOV

Побудуємо словник тільки за навчальним корпусом. Два спеціальні токени:

- `<pad>` доповнює послідовності до однакової довжини;
- `<unk>` замінює невідоме слово (**out of vocabulary**, OOV).

Якщо модель бачила `кіт`, але не бачила `котик`, усе слово `котик` стане
`<unk>`. Різні невідомі слова отримають однаковий id — їх уже неможливо
відрізнити.



**Ключове:** `<unk>` має ID 1. Два різні невідомі слова можуть дати однакові ID, і зміст втратиться. Embedding-таблиця має V × d параметрів: більший словник потребує більше пам’яті.

In [ ]:

def build_word_vocab(texts):
    tokens = sorted({token for text in texts for token in word_tokenize(text)})
    return {"<pad>": 0, "<unk>": 1, **{token: i for i, token in enumerate(tokens, 2)}}


def encode_words(text, vocab):
    return [vocab.get(token, vocab["<unk>"]) for token in word_tokenize(text)]


def test_word_vocab():
    vocab = build_word_vocab(["Кіт спить.", "Пес спить."])
    assert vocab["<pad>"] == 0
    assert vocab["<unk>"] == 1
    assert encode_words("Кіт біжить.", vocab) == [vocab["кіт"], 1, vocab["."]]
    assert encode_words("Хом'як біжить.", vocab)[0:2] == [1, 1]


if __name__ == "__main__":
    test_word_vocab()
    print("✓ word vocabulary")

## 2. Символьні n-грами

Замість цілого слова можна взяти всі послідовності з `n` сусідніх символів.
Додамо маркери `^` і `$`, щоб відрізняти початок та кінець слова:

```python
char_ngrams("кіт", 3) == ["^кі", "кіт", "іт$"]
```

У споріднених слів є спільні частини. Наприклад, навіть якщо `токенізація`
не траплялася в корпусі, її триграми можуть перетинатися з триграмами слова
`токенізатор`. Отже, замість одного беззмістовного `<unk>` ми отримуємо
часткову інформацію про невідоме слово.



**Пояснення:** Жаккар = кількість спільних унікальних n-грам / кількість усіх різних. Множина не враховує порядок і повторення. Для «кіт» із маркерами маємо `^кіт$` (5 символів): при n=6 повернеться порожній список, а не виняток. Для коротких слів можна використати кілька n або явно визначений fallback; тут збережено умову шаблону.

In [ ]:

def char_ngrams(word, n=3):
    if not isinstance(n, int) or isinstance(n, bool) or n <= 0:
        raise ValueError("n має бути додатним цілим числом")
    if not word:
        return []
    bounded = "^" + word + "$"
    return [bounded[i:i+n] for i in range(len(bounded) - n + 1)]


def ngram_overlap(word_a, word_b, n=3):
    a, b = set(char_ngrams(word_a, n)), set(char_ngrams(word_b, n))
    return len(a & b) / len(a | b) if a | b else 1.0


def test_ngrams():
    assert char_ngrams("кіт", 3) == ["^кі", "кіт", "іт$"]
    assert char_ngrams("кіт", 2) == ["^к", "кі", "іт", "т$"]
    assert char_ngrams("", 3) == []
    assert ngram_overlap("токенізатор", "токенізація", 3) > 0.4
    assert ngram_overlap("кіт", "трактор", 3) == 0.0

    try:
        char_ngrams("слово", 0)
    except ValueError:
        pass
    else:
        raise AssertionError("n має бути додатним")


if __name__ == "__main__":
    test_ngrams()
    print("✓ character n-grams")

## 3. Byte Pair Encoding (BPE)

Символьна токенізація не має OOV для знайомого алфавіту, але створює довгі
послідовності. Токенізація словами дає короткі послідовності, але величезний
словник та OOV. **BPE** шукає компроміс:

1. починаємо з окремих символів;
2. рахуємо частоти всіх сусідніх пар у корпусі;
3. найчастішу пару об'єднуємо в один токен;
4. повторюємо задану кількість разів.

Часті фрагменти стають одним токеном, а рідкі слова все одно можна скласти з
менших частин. Маркер `</w>` позначає кінець слова й не дає зливати символи
з різних слів.



**На захисті:** BPE обирає найчастішу пару з урахуванням повторів слів. `train_bpe` вивчає порядок злиттів, `apply_bpe` застосовує його до нового слова. Це дискретні підрахунки, без градієнтного спуску. Наш варіант працює із символами; byte-level BPE GPT стартує з байтів.

In [ ]:

def word_frequencies(texts):
    return Counter(token for text in texts for token in word_tokenize(text)
                   if token.replace("'", "").replace("’", "").replace("ʼ", "").isalpha())


def most_frequent_pair(vocabulary):
    counts = Counter()
    for symbols, frequency in vocabulary.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += frequency
    return min(counts, key=lambda pair: (-counts[pair], pair)) if counts else None


def merge_pair(symbols, pair):
    merged, i = [], 0
    while i < len(symbols):
        if i + 1 < len(symbols) and (symbols[i], symbols[i+1]) == pair:
            merged.append(symbols[i] + symbols[i+1])
            i += 2
        else:
            merged.append(symbols[i])
            i += 1
    return tuple(merged)


def train_bpe(texts, num_merges):
    if not isinstance(num_merges, int) or num_merges < 0:
        raise ValueError("num_merges має бути невід'ємним цілим")
    vocabulary = {tuple(word) + ("</w>",): count
                  for word, count in word_frequencies(texts).items()}
    merges = []
    for _ in range(num_merges):
        pair = most_frequent_pair(vocabulary)
        if pair is None:
            break
        updated = Counter()
        for symbols, count in vocabulary.items():
            updated[merge_pair(symbols, pair)] += count
        vocabulary = updated
        merges.append(pair)
    return merges



def apply_bpe(word, merges):
    if not word:
        return []
    symbols = tuple(word) + ("</w>",)
    for pair in merges:
        symbols = merge_pair(symbols, pair)
    return list(symbols)


def test_bpe():
    vocabulary = {
        ("м", "а", "м", "а", "</w>"): 2,
        ("м", "а", "л", "а", "</w>"): 1,
    }
    assert most_frequent_pair(vocabulary) == ("м", "а")
    assert merge_pair(("м", "а", "м", "а", "</w>"), ("м", "а")) == (
        "ма", "ма", "</w>"
    )

    merges = train_bpe(["мама мама мала тон"], 3)
    assert merges[0] == ("м", "а")
    assert apply_bpe("мама", merges) == ["мама</w>"]

    unseen = apply_bpe("мамонт", merges)
    assert "<unk>" not in unseen
    assert "".join(unseen).replace("</w>", "") == "мамонт"


if __name__ == "__main__":
    test_bpe()
    print("✓ BPE")

### Чи справді BPE скорочує послідовності?

Перевіримо це вимірюванням, а не припущенням. До навчання одне слово має
приблизно стільки токенів, скільки в ньому символів. Після кожного злиття
деякі часті пари займають одну позицію замість двох.

Важливо: більше злиттів зазвичай означає коротші послідовності, але більший
словник. BPE не гарантує, що кожне окреме слово стане коротшим, особливо якщо
воно не схоже на навчальний корпус.



**Висновок:** більше злиттів скорочує представлення корпусу ціною більшого словника. Ця функція рахує лише літерні слова, тому її значення не тотожне fertility для всього тексту нижче.

In [ ]:

def average_bpe_length(texts, merges):
    frequencies = word_frequencies(texts)
    total_words = sum(frequencies.values())
    if not total_words:
        raise ValueError("Корпус без слів")
    return sum(count * len(apply_bpe(word, merges))
               for word, count in frequencies.items()) / total_words


def test_average_bpe_length():
    corpus = [
        "токен токени токенізація",
        "токенізатор токенізує текст",
    ]
    merges = train_bpe(corpus, 20)
    before = average_bpe_length(corpus, [])
    after = average_bpe_length(corpus, merges)
    assert after < before
    assert before > 1


if __name__ == "__main__":
    test_average_bpe_length()
    print("✓ BPE reduces average sequence length")

## 4. WordPiece

WordPiece навчається майже як BPE: починає із символів і поступово зливає
сусідні фрагменти. Значну частину логіки можна адаптувати з попереднього завдання.

Відмінності: `##` позначає продовження слова; пари вибираються за оцінкою,
а не лише за частотою; під час токенізації шукають найдовший фрагмент зі словника.

`score(a, b) = frequency(a, b) / (frequency(a) * frequency(b))`



**Різниця з BPE:** score = f(a,b)/(f(a)f(b)); `##` позначає продовження слова. При застосуванні WordPiece шукає найдовший фрагмент зі словника. Якщо повністю розібрати слово не вдається, повертається один `[UNK]` для всього слова.

In [ ]:


def wordpiece_symbols(word):
    return tuple(character if i == 0 else "##" + character
                 for i, character in enumerate(word))


def wordpiece_pair_scores(vocabulary):
    token_counts, pair_counts = Counter(), Counter()
    for symbols, count in vocabulary.items():
        for symbol in symbols:
            token_counts[symbol] += count
        for pair in zip(symbols, symbols[1:]):
            pair_counts[pair] += count
    return {pair: count / (token_counts[pair[0]] * token_counts[pair[1]])
            for pair, count in pair_counts.items()}


def merge_wordpiece_pair(symbols, pair):
    merged, i = [], 0
    while i < len(symbols):
        if i + 1 < len(symbols) and (symbols[i], symbols[i+1]) == pair:
            right = symbols[i+1][2:] if symbols[i+1].startswith("##") else symbols[i+1]
            merged.append(symbols[i] + right)
            i += 2
        else:
            merged.append(symbols[i])
            i += 1
    return tuple(merged)


def train_wordpiece(texts, vocab_size):
    if not isinstance(vocab_size, int) or vocab_size <= 0:
        raise ValueError("vocab_size має бути додатним цілим")
    vocabulary = {wordpiece_symbols(word): count
                  for word, count in word_frequencies(texts).items()}
    vocab = {"[UNK]"} | {symbol for symbols in vocabulary for symbol in symbols}
    if len(vocab) > vocab_size:
        raise ValueError("vocab_size менший за початковий WordPiece-алфавіт")
    while len(vocab) < vocab_size:
        scores = wordpiece_pair_scores(vocabulary)
        if not scores:
            break
        pair = min(scores, key=lambda pair: (-scores[pair], pair))
        joined = merge_wordpiece_pair(pair, pair)[0]
        updated = Counter()
        for symbols, count in vocabulary.items():
            updated[merge_wordpiece_pair(symbols, pair)] += count
        vocabulary = updated
        vocab.add(joined)
    return vocab


def apply_wordpiece(word, vocab, unk_token="[UNK]"):
    pieces, start = [], 0
    while start < len(word):
        end = len(word)
        while end > start:
            candidate = ("##" if start else "") + word[start:end]
            if candidate in vocab:
                break
            end -= 1
        if end == start:
            return [unk_token]
        pieces.append(candidate)
        start = end
    return pieces


def test_wordpiece():
    weighted_vocab = {
        ("а", "##б"): 2,
        ("а", "##в"): 1,
        ("г", "##в"): 2,
    }
    scores = wordpiece_pair_scores(weighted_vocab)
    assert scores[("а", "##б")] > scores[("а", "##в")]
    assert merge_wordpiece_pair(("м", "##а", "##м", "##а"), ("м", "##а")) == (
        "ма", "##м", "##а"
    )

    vocab = train_wordpiece(["мама мама мала тон"], 30)
    assert "[UNK]" in vocab
    assert apply_wordpiece("мама", vocab) == ["мама"]

    bert_vocab = {"[UNK]", "ток", "токен", "##ен", "##ізація"}
    assert apply_wordpiece("токенізація", bert_vocab) == ["токен", "##ізація"]
    assert apply_wordpiece("токенx", bert_vocab) == ["[UNK]"]
    assert apply_wordpiece("", bert_vocab) == []

    try:
        train_wordpiece(["текст"], 0)
    except ValueError:
        pass
    else:
        raise AssertionError("vocab_size має бути додатним")


if __name__ == "__main__":
    test_wordpiece()
    print("✓ WordPiece")

## 5. Великий експеримент: FLORES та GPT

Тепер застосуємо вивчені ідеї до справжнього багатомовного корпусу. Навчимо
власні токенізатори на англійській, українській та грузинській мовах, а потім
порівняємо їх із готовими токенізаторами GPT.

Використаємо `dev` split FLORES-200: по 997 паралельних речень кожною мовою,
тобто 2991 навчальний рядок. Це небагато для реальної мовної моделі, але
достатньо для навчального експерименту з токенізацією.

Код завантаження даних уже готовий. Його реалізовувати не потрібно.



**Дані:** 997 речень × 3 мови = 2991 навчальний рядок. У дзеркалі потрібний FLORES dev названо `validation`. Тут навчаємо лише tokenizer-и, а не GPT.

In [ ]:

from datasets import load_dataset


FLORES_LANGUAGES = {
    "English": "eng_Latn",
    "Українська": "ukr_Cyrl",
    "ქართული": "kat_Geor",
}


def load_flores_dev():
    """Завантажує по 997 речень FLORES для кожної вибраної мови."""
    corpora = {}
    for language, config in FLORES_LANGUAGES.items():
        dataset = load_dataset(
            "tomasmajercik/flores-parquet",
            config,
            split="validation",
        )
        corpora[language] = list(dataset["sentence"])
    return corpora


flores = load_flores_dev()

assert all(len(texts) == 997 for texts in flores.values())
train_texts = [
    text
    for language_texts in flores.values()
    for text in language_texts
]

print({language: len(texts) for language, texts in flores.items()})
print("Усього навчальних рядків:", len(train_texts))

Подивіться на одне й те саме речення трьома мовами. FLORES є паралельним
корпусом, тому рядки з однаковим індексом мають однаковий зміст.



**Чому однакова кількість речень:** кожна мова має однакову кількість прикладів у спільному навчальному корпусі. Це зменшує домінування через обсяг даних. Але рівна кількість речень не вирівнює слова, символи чи частку словника. Паралельні переклади додатково дають відповідний зміст; 997 речень не є повним представленням мови.

In [ ]:

example_index = 0
for language, texts in flores.items():
    print(f"{language}: {texts[example_index]}\n")

### Навчання на FLORES із різними словниками

Тут не створюємо нових алгоритмів. Повторно використайте вже реалізовані
`train_bpe`, `apply_bpe`, `train_wordpiece` та `apply_wordpiece`.

Навчіть обидва токенізатори зі словниками приблизно 500, 1000 і 2000 токенів.
Для BPE початковий словник уже містить символи, а кожне merge додає один
токен, тому бажаний розмір словника треба перетворити на кількість merges.



**Бюджет:** BPE вже має початковий алфавіт і `</w>`, тому num_merges = vocab_size − initial_vocab_size. Токени не зобов’язані ділитися між мовами порівну. Наївне навчання на CPU може займати багато хвилин; під час захисту можна показувати збережені результати.

In [ ]:

VOCAB_SIZES = (500, 1000, 2000)


def bpe_num_merges(texts, vocab_size):
    """Обчислює кількість BPE-merges для бажаного розміру словника."""
    frequencies = word_frequencies(texts)
    characters = {
        character
        for word in frequencies
        for character in word
    }
    initial_vocab_size = len(characters) + 1
    if vocab_size < initial_vocab_size:
        raise ValueError("vocab_size менший за початковий алфавіт")
    return vocab_size - initial_vocab_size


def apply_bpe_to_text(text, merges):
    """Застосовує створений раніше BPE до всіх токенів тексту."""
    return [
        piece
        for token in word_tokenize(text)
        for piece in apply_bpe(token, merges)
    ]


def apply_wordpiece_to_text(text, vocab):
    """Застосовує створений раніше WordPiece до всіх токенів тексту."""
    return [
        piece
        for token in word_tokenize(text)
        for piece in apply_wordpiece(token, vocab)
    ]


bpe_models = {}
wordpiece_models = {}

for vocab_size in VOCAB_SIZES:
    bpe_models[vocab_size] = train_bpe(
        train_texts,
        bpe_num_merges(train_texts, vocab_size),
    )
    wordpiece_models[vocab_size] = train_wordpiece(
        train_texts,
        vocab_size,
    )

print("Навчено BPE і WordPiece для:", VOCAB_SIZES)

### Готові GPT-токенізатори

Наступний код уже готовий. `tiktoken` імпортує ті самі encoding-и, які
використовують відповідні покоління GPT. GPT-3.5 і класичний GPT-4 мають
однаковий `cl100k_base`, тому їхні результати збігатимуться.



**Важливо:** це порівняння encoding-ів, а не якості генерації GPT. GPT-3.5 і класичний GPT-4 у шаблоні мають однаковий cl100k_base, тому дадуть однакове розбиття. API-запити не виконуються.

In [ ]:

import tiktoken


gpt_encodings = {
    "GPT-2": tiktoken.encoding_for_model("gpt-2"),
    "GPT-3.5": tiktoken.encoding_for_model("gpt-3.5-turbo"),
    "GPT-4": tiktoken.encoding_for_model("gpt-4"),
    "GPT-5": tiktoken.encoding_for_model("gpt-5"),
}

for name, encoding in gpt_encodings.items():
    print(f"{name:7} -> {encoding.name:12} ({encoding.n_vocab} tokens)")

### Fertility

**Fertility** — середня кількість токенів на одне слово:

$$\text{fertility} =
\frac{\text{загальна кількість токенів}}
     {\text{загальна кількість слів}}$$

У цій лабораторній словом вважаємо непорожній фрагмент після `str.split()`.
Рахуйте відношення двох сум для всього корпусу, а не середнє значень окремих
речень: тоді кожне слово має однакову вагу.



**Правильний розрахунок:** ділимо дві суми для всього корпусу, а не усереднюємо частки речень. У тесті list("aa b") має 4 символи разом із пробілом, list("ccc") - 3; слів 2 + 1, отже fertility = 7/3.

In [ ]:

def fertility(tokenize, texts):
    texts = list(texts)
    words = sum(len(text.split()) for text in texts)
    if not words:
        raise ValueError("Корпус без слів має давати ValueError")
    return sum(len(tokenize(text)) for text in texts) / words


def test_fertility():
    assert fertility(list, ["aa b", "ccc"]) == 7 / 3

    try:
        fertility(list, ["", "   "])
    except ValueError:
        pass
    else:
        raise AssertionError("Корпус без слів має давати ValueError")


if __name__ == "__main__":
    test_fertility()
    print("✓ fertility")

### Порівняння за мовами

Побудуйте таблицю fertility для всіх 10 токенізаторів:

- BPE: 500, 1000, 2000;
- WordPiece: 500, 1000, 2000;
- GPT-2, GPT-3.5, GPT-4, GPT-5.

Наші моделі перевіряються на тому самому `dev`, на якому навчалися. Отже, це
train fertility — вона показує стиснення корпусу, але не узагальнення.



**Що пояснити:** на попередньому повному запуску GPT-2 мав найбільшу fertility для грузинської (21.545); для української 8.024, англійської 1.228. BPE-2000 для української дав 3.185, WordPiece-2000 - 5.412. Навчальний WordPiece за score може додавати рідкісні фрагменти й слабше зменшувати середню довжину. Це train fertility: дані навчання й оцінки ті самі. Низьке число через `[UNK]` не означає збереження інформації.

In [ ]:

if __name__ == "__main__":
    tokenizers_to_compare = {}

    for vocab_size, merges in bpe_models.items():
        tokenizers_to_compare[f"BPE-{vocab_size}"] = (
            lambda text, merges=merges: apply_bpe_to_text(text, merges)
        )

    for vocab_size, vocab in wordpiece_models.items():
        tokenizers_to_compare[f"WordPiece-{vocab_size}"] = (
            lambda text, vocab=vocab: apply_wordpiece_to_text(text, vocab)
        )

    for name, encoding in gpt_encodings.items():
        tokenizers_to_compare[name] = encoding.encode


    def fertility_table(tokenizers, corpora):
        return {name: {language: fertility(tokenize, texts)
                       for language, texts in corpora.items()}
                for name, tokenize in tokenizers.items()}


    results = fertility_table(tokenizers_to_compare, flores)

    header = f"{'tokenizer':<24}" + "".join(
        f"{language:>14}" for language in flores
    )
    print(header)
    print("-" * len(header))
    for tokenizer_name, language_values in results.items():
        row = f"{tokenizer_name:<24}" + "".join(
            f"{language_values[language]:14.3f}" for language in flores
        )
        print(row)

### Змішані та складні випадки

Середня fertility приховує конкретні невдалі розбиття. Порівняйте кількість
та самі токени для code-switching, різних систем письма, чисел, emoji, URL і
слів, яких не було у FLORES.



**Один приклад для захисту:** оберіть змішаний текст і порівняйте кількість токенів та `[UNK]`. GPT-2 сильніше дробить українську через словник під англомовні частоти; також впливають письмо, байти UTF-8 та словоформи. Всю різницю не можна пояснити тільки відмінками. Fertility доповнюють перевіркою покриття й точного відновлення тексту.

In [ ]:

MIXED_CASES = [
    "Я навчаю tokenizer на FLORES dataset.",
    "The model розуміє ქართულ ენას?",
    "Київ -> Tbilisi -> თბილისი -> London 🌍",
    "Замовлення №2026-09-24 коштує $19.99.",
    "support@example.com або https://example.com/допомога",
    "я дуже втомився від курсу основи Large Language Models"
]


def compare_mixed_cases(texts, tokenizers):
    for text in texts:
        print("\nТекст:", text)
        for name, tokenize in tokenizers.items():
            tokens = tokenize(text)
            print(f"  {name:18} {len(tokens):3} токенів; {tokens[:12]}")


if __name__ == "__main__":
    compare_mixed_cases(MIXED_CASES, tokenizers_to_compare)